<a href="https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: Click capture by position tier

Source: Finding 3, page 8.

The paper reports pooled CTR of 0.423% for positions 1–3, compared with 0.050% in its deepest position tier. CTR is calculated as total clicks divided by total impressions, rather than an average of individual row rates.

My methodology question: How much does this pattern persist within individual clients and comparable query, device, and country groups? Reporting client-level results and uncertainty would help establish whether a few high-impression clients dominate the pooled comparison.

The outcome is measured click-through rate, not a label confirming that editing a page will improve it. The comparison supports a descriptive association between position tiers and CTR. It does not by itself establish the effect of moving a page or revising its search appearance.

Lesson for my project: Compare CTR within relevant peer groups, report coverage, and avoid treating a peer gap as recoverable clicks.

### Finding 2: Freshness and traffic growth

Source: Finding 4, page 9.

The paper reports a growth-to-decline ratio of 7.88:1 for content updated 31–90 days earlier. It explicitly cautions against interpreting the 361+ day bucket’s 283:1 ratio as a stable headline because it contains only one declining page.

My methodology question: How are growing and declining labels defined, and how do their measurement windows align with update dates? A matched before-and-after comparison with comparable unupdated pages, accounting for prior traffic, content age, client, and seasonality, would help distinguish update-associated patterns from selection effects.

The paper discloses its observational design. Its comparisons do not establish that an update caused growth: editors may preferentially update promising pages, and the groups may differ before any update.

Lesson for my project: Keep input and outcome windows separate, explain proxy labels, report small denominators, and distinguish measured associations from intervention benefits.

### Overall approach

These questions seek clarification and stronger validation, rather than dismissing the paper. Its stated limitations are useful examples of the careful language I should apply to my own results.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

My Week 5 evaluation already separated clients between training and testing. Its limitation was reliance on one small test set: 246 items from 6 clients. I extended the evaluation to five-fold GroupKFold, producing an out-of-fold prediction for every one of the 6,531 eligible items across 21 clients.

Each fold fits preprocessing and logistic regression using training items only. Client overlap is checked explicitly. The features, model settings, outcome definition, and Week 4 scoring formula remain unchanged.

On the original split, both methods achieved Precision@10 of 1.00. The baseline achieved average precision 0.8399 and ROC-AUC 0.7321; logistic regression achieved 0.8648 and 0.7711.

In pooled out-of-fold evaluation, both achieved Precision@10 and Precision@20 of 1.00. The baseline achieved average precision 0.8566 and ROC-AUC 0.7840, compared with 0.8025 and 0.6787 for logistic regression. The pooled positive-label base rate was 0.6567.

The model’s broader-ranking advantage on the original split did not persist in the pooled evaluation. Individual-fold results vary, and neither method dominates every metric in every fold. Fold 1 contains 4,273 items from one client, so pooled results are strongly influenced by that client.

Before and after use different evaluation populations; their difference cannot be attributed solely to the validation change. Pooled probabilities also come from different fitted models and may differ in calibration.

The frozen peer benchmarks include all eligible early-window clients. This remains a known-reference-population experiment, not fully inductive validation on unseen clients. Outcome-coverage filtering further restricts the evaluable population. No later outcomes enter model inputs or benchmark construction. June remains untouched.

These findings support retaining the simple baseline. They do not establish content-update benefit or human-confirmed review quality.

In [7]:
%pip -q install huggingface_hub duckdb scikit-learn

import pandas as pd
import numpy as np
import duckdb

from google.colab import userdata
from huggingface_hub import hf_hub_download

token = userdata.get("HF_TOKEN").strip()

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
    repo_type="dataset",
    token=token
)

con = duckdb.connect()
con.read_parquet(
    march_path,
    hive_partitioning=True
).create_view("march_raw")

print("March dataset ready.")

display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_raw
""").df())


March dataset ready.


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [8]:
# Reconstruct the Week 4 early-window table.
early = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS observed_days,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) * 1.0
            / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,

        SUM(
            CASE WHEN gsc_avg_position > 0
                 THEN gsc_avg_position * gsc_impressions END
        ) / NULLIF(
            SUM(
                CASE WHEN gsc_avg_position > 0
                     THEN gsc_impressions END
            ), 0
        ) AS avg_position

    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) >= 10
       AND SUM(gsc_impressions) > 0
""").df()

early["position_bucket"] = pd.cut(
    early["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1–3", ">3–10", ">10–20", ">20"]
)

early["volume_bucket"] = pd.cut(
    early["impressions"],
    bins=[99, 499, 1999, float("inf")],
    labels=["100–499", "500–1,999", "2,000+"]
)

peer_stats = (
    early.loc[
        (early["impressions"] >= 100)
        & early["position_bucket"].notna()
    ]
    .groupby(
        ["position_bucket", "volume_bucket"],
        observed=True
    )
    .agg(
        peer_count=("ctr", "size"),
        peer_median_ctr=("ctr", "median")
    )
    .reset_index()
)

early = early.merge(
    peer_stats,
    on=["position_bucket", "volume_bucket"],
    how="left",
    validate="many_to_one"
)

eligible = (
    (early["impressions"] >= 100)
    & early["position_bucket"].notna()
    & (early["peer_count"] >= 20)
    & early["ctr"].notna()
    & early["peer_median_ctr"].notna()
)

early["baseline_score"] = np.where(
    eligible,
    early["impressions"]
    * (early["peer_median_ctr"] - early["ctr"]).clip(lower=0),
    0.0
)

early["baseline_eligible"] = eligible

assert not early.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()

print("Early-window items:", len(early))
print(
    "Positive baseline scores:",
    (early["baseline_score"] > 0).sum()
)
print(
    "Items meeting baseline scoring requirements:",
    eligible.sum()
)

Early-window items: 6607
Positive baseline scores: 3280
Items meeting baseline scoring requirements: 6565


In [9]:
# Later data constructs the outcome, never the model inputs.
late = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS outcome_days,
        SUM(gsc_impressions) AS late_impressions,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS late_ctr
    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

experiment = early.loc[
    early["baseline_eligible"]
].merge(
    late,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

experiment = experiment.loc[
    (experiment["outcome_days"] >= 5)
    & (experiment["late_impressions"] >= 100)
    & experiment["late_ctr"].notna()
].copy()

experiment["later_below_peer"] = (
    experiment["late_ctr"] < experiment["peer_median_ctr"]
).astype(int)

experiment = experiment.sort_values(
    ["client_hash_id", "content_hash_id"]
).reset_index(drop=True)

feature_columns = [
    "impressions",
    "clicks",
    "ctr",
    "sessions",
    "engagement_rate"
]

assert not experiment.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()

print("Comparison items:", len(experiment))
print("Clients:", experiment["client_hash_id"].nunique())

display(
    experiment["later_below_peer"]
    .value_counts()
    .sort_index()
    .rename_axis("later_below_peer")
    .to_frame("item_count")
)

print("Missing model inputs:")
display(
    experiment[feature_columns]
    .isna().sum().to_frame("missing_count")
)

Comparison items: 6531
Clients: 21


,item_count
later_below_peer,
0,2242
1,4289


Missing model inputs:


,missing_count
impressions,0
clicks,0
ctr,0
sessions,0
engagement_rate,0


In [10]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

X = experiment[feature_columns].copy()
y = experiment["later_below_peer"].astype(int)
groups = experiment["client_hash_id"]
baseline = experiment["baseline_score"].to_numpy()

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    k = min(k, len(order))
    return float(np.asarray(labels)[order[:k]].mean())

def metrics(stage, method, labels, scores):
    labels = np.asarray(labels)
    return {
        "stage": stage,
        "method": method,
        "items": len(labels),
        "base_rate": labels.mean(),
        "precision_at_10": precision_at_k(labels, scores, 10),
        "precision_at_20": precision_at_k(labels, scores, 20),
        "average_precision": average_precision_score(labels, scores),
        "roc_auc": (
            roc_auc_score(labels, scores)
            if len(np.unique(labels)) == 2 else np.nan
        )
    }

# BEFORE: reproduce the original Week 5 split.
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)
train_idx, test_idx = next(splitter.split(X, y, groups))

assert set(groups.iloc[train_idx]).isdisjoint(
    set(groups.iloc[test_idx])
)

original_model = make_model()
original_model.fit(X.iloc[train_idx], y.iloc[train_idx])
original_scores = original_model.predict_proba(
    X.iloc[test_idx]
)[:, 1]

results = [
    metrics(
        "Before: original split", "Week 4 baseline",
        y.iloc[test_idx], baseline[test_idx]
    ),
    metrics(
        "Before: original split", "Logistic regression",
        y.iloc[test_idx], original_scores
    )
]

# AFTER: each client is held out in exactly one fold.
gkf = GroupKFold(n_splits=5)
oof_scores = np.full(len(X), np.nan)
fold_number = np.full(len(X), -1)
fold_rows = []
validation_models = []

for fold, (train, test) in enumerate(
    gkf.split(X, y, groups), start=1
):
    assert set(groups.iloc[train]).isdisjoint(
        set(groups.iloc[test])
    )
    assert y.iloc[train].nunique() == 2

    fitted = make_model()
    fitted.fit(X.iloc[train], y.iloc[train])
    probabilities = fitted.predict_proba(X.iloc[test])[:, 1]

    oof_scores[test] = probabilities
    fold_number[test] = fold
    validation_models.append(fitted)

    for name, scores in [
        ("Week 4 baseline", baseline[test]),
        ("Logistic regression", probabilities)
    ]:
        row = metrics(
            f"Fold {fold}", name, y.iloc[test], scores
        )
        row["test_clients"] = groups.iloc[test].nunique()
        fold_rows.append(row)

assert np.isfinite(oof_scores).all()
assert (fold_number > 0).all()

results.extend([
    metrics(
        "After: pooled out-of-fold", "Week 4 baseline",
        y, baseline
    ),
    metrics(
        "After: pooled out-of-fold", "Logistic regression",
        y, oof_scores
    )
])

comparison = pd.DataFrame(results)
fold_results = pd.DataFrame(fold_rows)

print("Before/after comparison:")
display(comparison.round(4))

print("Individual fold results:")
display(fold_results.round(4))

Before/after comparison:


,stage,method,items,base_rate,precision_at_10,precision_at_20,average_precision,roc_auc
0,Before: original split,Week 4 baseline,246,0.6911,1.0,1.00,0.8399,0.7321
1,Before: original split,Logistic regression,246,0.6911,1.0,0.95,0.8648,0.7711
2,After: pooled out-of-fold,Week 4 baseline,6531,0.6567,1.0,1.00,0.8566,0.7840
3,After: pooled out-of-fold,Logistic regression,6531,0.6567,1.0,1.00,0.8025,0.6787


Individual fold results:


,stage,method,items,base_rate,precision_at_10,precision_at_20,average_precision,roc_auc,test_clients
0,Fold 1,Week 4 baseline,4273,0.6293,1.0,0.95,0.8500,0.7985,1
1,Fold 1,Logistic regression,4273,0.6293,1.0,1.00,0.8431,0.7669,1
2,Fold 2,Week 4 baseline,647,0.8068,1.0,1.00,0.9232,0.7964,1
3,Fold 2,Logistic regression,647,0.8068,1.0,1.00,0.9180,0.7522,1
4,Fold 3,Week 4 baseline,634,0.6278,1.0,1.00,0.8155,0.7488,1
5,Fold 3,Logistic regression,634,0.6278,1.0,0.95,0.8441,0.7803,1
6,Fold 4,Week 4 baseline,489,0.6564,1.0,1.00,0.8422,0.7638,6
7,Fold 4,Logistic regression,489,0.6564,1.0,1.00,0.8480,0.7597,6
8,Fold 5,Week 4 baseline,488,0.7357,1.0,1.00,0.8772,0.7545,12
9,Fold 5,Logistic regression,488,0.7357,1.0,0.95,0.8907,0.7763,12


## 3. Leakage audit


### Leakage checks

Only impressions, clicks, CTR, sessions, and engagement rate enter the model. These features use 1–20 March observations. The label uses CTR from 21–31 March, compared with the frozen early-window peer benchmark.

Explicit assertions exclude later measurements, outcome labels, baseline scores, peer benchmarks, and identifiers from model inputs. Preprocessing is fitted separately within each training fold. Client overlap between training and validation is checked, and every item receives an out-of-fold prediction.

These checks address direct feature leakage but do not remove every methodological limitation. Peer benchmarks include validation clients’ early observations, so validation is not fully inductive. Eligibility also depends on sufficient later observations; results apply only to that retrospectively selected population.

Clicks, impressions, and CTR are mathematically related. Their relationship is not future leakage, but it complicates coefficient interpretation. Actual reporting delays would need checking before deployment.

### Failure examples

There are 1,979 out-of-fold classification errors among 6,531 items at threshold 0.5. These errors supplement the ranking metrics; the threshold does not determine the top-10 queue.

The three most confident wrong predictions are false positives in Fold 1:

- Case 2883: 2,061 early impressions and zero early clicks. Predicted probability is 0.99887, but later CTR is 0.00213, above its benchmark of 0.00159.
- Case 2680: 2,034 early impressions and zero early clicks. Predicted probability is 0.99885, but later CTR is 0.00242, above the same benchmark.
- Case 1180: 2,337 early impressions and zero early clicks. Predicted probability is 0.99852, but later CTR is 0.00252, above the same benchmark.

CTR values are fractions. These examples show that zero early clicks do not guarantee continued below-benchmark CTR. The model makes extremely confident mistakes when later performance improves. The data does not establish what caused the improvement.

Error rates vary across folds: 32.53%, 23.96%, 26.81%, 30.88%, and 23.16%. Fold sizes and client composition differ, so these are descriptive comparisons rather than controlled explanations of error.

In [11]:
# Explicitly check which columns enter the model.
expected_features = [
    "impressions",
    "clicks",
    "ctr",
    "sessions",
    "engagement_rate"
]

assert list(X.columns) == expected_features

forbidden_features = {
    "later_below_peer",
    "late_ctr",
    "late_impressions",
    "outcome_days",
    "baseline_score",
    "peer_median_ctr",
    "client_hash_id",
    "content_hash_id"
}

assert forbidden_features.isdisjoint(X.columns)
assert np.isfinite(oof_scores).all()

print("Feature allowlist passed.")
print("No outcome, baseline score, benchmark, or ID columns in X.")
print("Every item has an out-of-fold prediction.")

# Examine classification errors at a fixed 0.5 threshold.
audit_cases = experiment.copy()
audit_cases["case_number"] = np.arange(1, len(audit_cases) + 1)
audit_cases["fold"] = fold_number
audit_cases["model_probability"] = oof_scores
audit_cases["predicted_label"] = (oof_scores >= 0.5).astype(int)

audit_cases["wrong"] = (
    audit_cases["predicted_label"]
    != audit_cases["later_below_peer"]
)

audit_cases["error_magnitude"] = abs(
    audit_cases["model_probability"]
    - audit_cases["later_below_peer"]
)

print("Out-of-fold errors at threshold 0.5:",
      int(audit_cases["wrong"].sum()))

wrong_cases = (
    audit_cases.loc[audit_cases["wrong"]]
    .sort_values("error_magnitude", ascending=False, kind="stable")
    .head(3)
)

print("Three most confident wrong predictions:")
display(wrong_cases[[
    "case_number",
    "fold",
    "impressions",
    "clicks",
    "ctr",
    "sessions",
    "engagement_rate",
    "peer_median_ctr",
    "late_ctr",
    "later_below_peer",
    "model_probability",
    "predicted_label"
]].round(5))

print("Errors by fold:")
display(
    audit_cases.groupby("fold")
    .agg(
        items=("wrong", "size"),
        error_rate=("wrong", "mean"),
        proxy_positive_rate=("later_below_peer", "mean")
    ).round(4)
)


Feature allowlist passed.
No outcome, baseline score, benchmark, or ID columns in X.
Every item has an out-of-fold prediction.
Out-of-fold errors at threshold 0.5: 1979
Three most confident wrong predictions:


,case_number,fold,impressions,clicks,ctr,sessions,engagement_rate,peer_median_ctr,late_ctr,later_below_peer,model_probability,predicted_label
2882,2883,1,2061.0,0.0,0.0,30.0,0.0,0.00159,0.00213,0,0.99887,1
2679,2680,1,2034.0,0.0,0.0,30.0,0.0,0.00159,0.00242,0,0.99885,1
1179,1180,1,2337.0,0.0,0.0,284.0,0.0,0.00159,0.00252,0,0.99852,1


Errors by fold:


,items,error_rate,proxy_positive_rate
fold,,,
1,4273,0.3253,0.6293
2,647,0.2396,0.8068
3,634,0.2681,0.6278
4,489,0.3088,0.6564
5,488,0.2316,0.7357


## 4. Claim rewrite

### Earlier finding

In the original Week 5 client-grouped split, logistic regression had higher average precision and ROC-AUC than the Week 4 baseline. It did not improve Precision@10.

### Overstatement to avoid

“The model is better than the baseline and identifies pages that will benefit from content updates.”

This statement is not supported. It generalizes from one split, ignores metrics where the model did not win, and treats a CTR proxy as evidence of update benefit.

### Revised claim after validation

In this March 2026 eligible subset, both methods achieved Precision@10 of 1.00 in the original split and pooled five-fold evaluation. The model’s original advantage in broader ranking metrics did not persist in the pooled results: the baseline achieved average precision 0.8566 and ROC-AUC 0.7840, versus 0.8025 and 0.6787 for logistic regression.

Individual-fold results varied. The evaluation used a known early-window reference population, and pooled results were strongly influenced by one large client. These measured findings support retaining the simpler baseline for this exploratory workflow.

The ranked output provides directional decision support for investigating below-peer CTR. It does not establish human-confirmed review value, causal effects, content-update benefit, or performance on a fully unseen future population.

### What would strengthen the evidence

Further evaluation should use independent time windows, benchmarks constructed under the intended deployment conditions, and client-level summaries alongside item-level metrics. Independent page-review labels would be needed to evaluate the original editorial decision. Measuring update benefit would require an appropriate intervention study.

June remains unused. No sealed-test result is claimed.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.